# Live outputs, views, and canvas

Copy this notebook into a disposable JupyterLab project. Run one call, let its Media status finish, then run the following inspection cell. The source output fixtures stay in this notebook, including unsaved changes. Browser capture needs this repository build; PyPI 0.1.15 does not include it.

In [ ]:
from io import BytesIO
from PIL import Image as PILImage
from IPython.display import display, Image, SVG, HTML
from nbinlineai.tools import (read_notebook_view, read_selection, list_outputs, read_output, export_output, list_canvases, capture_canvas, export_canvas, start_canvas, capture_notebook_region)
image = PILImage.new('RGB', (8, 8), '#d4402f')
stream = BytesIO(); image.save(stream, format='PNG')
PNG_BYTES = stream.getvalue()
SVG_MARKUP = '<svg xmlns="http://www.w3.org/2000/svg" width="24" height="18"><rect width="24" height="18" fill="#2542a3"/></svg>'
print('Disposable PNG and original SVG are ready; no file was saved.')

In [ ]:
display(Image(data=PNG_BYTES))

In [ ]:
print('Existing output text: red square and blue vector')

In [ ]:
display(SVG(data=SVG_MARKUP))

## Inspect the existing notebook

The first five tools work with live model state. They do not run an output cell for you. The IDs below belong to this copied notebook.

In [ ]:
view = read_notebook_view()

In [ ]:
print(view.status, view.result, view.error)

In [ ]:
selection = read_selection(max_chars=120)

In [ ]:
print(selection.status, selection.result, selection.error)
print('An empty selection is expected unless this notebook editor held focus and selected text.')

In [ ]:
raster_listing = list_outputs('output-raster')
text_listing = list_outputs('output-text')
vector_listing = list_outputs('output-vector')

In [ ]:
for label, receipt in [('raster', raster_listing), ('text', text_listing), ('vector', vector_listing)]: print(label, receipt.status, receipt.result, receipt.error)
raster_ref = raster_listing.result['outputs'][0] if raster_listing.result else None
text_ref = text_listing.result['outputs'][0] if text_listing.result else None
vector_ref = vector_listing.result['outputs'][0] if vector_listing.result else None

In [ ]:
text_read = read_output(text_ref['cell_id'], text_ref['output_id'], text_ref['revision'], max_chars=120) if text_ref else None

In [ ]:
print(text_read.status, text_read.result, text_read.error) if text_read else print('List the text output first.')

In [ ]:
raster_export = export_output(raster_ref['cell_id'], raster_ref['output_id'], raster_ref['revision']) if raster_ref else None
vector_export = export_output(vector_ref['cell_id'], vector_ref['output_id'], vector_ref['revision'], mime='image/svg+xml') if vector_ref else None

In [ ]:
print('Raster:', raster_export.status, type(raster_export.result).__name__, getattr(raster_export.result, 'size', None), raster_export.error) if raster_export else print('List raster output first.')
print('SVG:', vector_export.status, vector_export.result == SVG_MARKUP, vector_export.error) if vector_export else print('List SVG output first.')

## An already-rendered canvas

The plain HTML fixture is deliberately small and contains no script or app adapter. Some JupyterLab sanitizers remove canvas HTML; `list_canvases` then reports unavailable. To verify colored pixels, draw into this existing canvas using a disposable browser test or a trusted notebook renderer before capture. No tool here executes drawing code.

In [ ]:
display(HTML('<canvas width="8" height="8" aria-label="Disposable canvas"></canvas>'))

In [ ]:
canvas_listing = list_outputs('output-canvas')

In [ ]:
print(canvas_listing.status, canvas_listing.result, canvas_listing.error)
canvas_output_ref = canvas_listing.result['outputs'][0] if canvas_listing.result and canvas_listing.result['outputs'] else None

In [ ]:
canvases = list_canvases(canvas_output_ref['cell_id'], canvas_output_ref['output_id'], canvas_output_ref['revision']) if canvas_output_ref else None

In [ ]:
print(canvases.status, canvases.result, canvases.error) if canvases else print('List the canvas output first.')
canvas_ref = canvases.result['canvases'][0] if canvases and canvases.result and canvases.result.get('canvases') else None

In [ ]:
canvas_still = capture_canvas(canvas_ref) if canvas_ref else None

In [ ]:
print(canvas_still.status, getattr(canvas_still.result, 'size', None), canvas_still.error) if canvas_still else print('Canvas unavailable in this renderer.')

In [ ]:
canvas_saved = export_canvas(canvas_ref) if canvas_ref else None

In [ ]:
print(canvas_saved.status, canvas_saved.media, canvas_saved.error) if canvas_saved else print('Canvas unavailable in this renderer.')

In [ ]:
canvas_source = start_canvas(canvas_ref, frame_rate=12) if canvas_ref else None

In [ ]:
print(canvas_source.status, canvas_source.result, canvas_source.error) if canvas_source else print('Canvas unavailable in this renderer.')
print('The cleanup cell below stops an opened source through the shared source control.')

## Capture a supported visible output region

Scroll so the raster output below is completely visible. The region includes its displayed output rectangle only, not code or inputs. Unsupported or partly hidden output is an explicit error.

In [ ]:
display(Image(data=PNG_BYTES))

In [ ]:
region = capture_notebook_region(['region-raster'])

In [ ]:
print(region.status, getattr(region.result, 'size', None), region.error)

In [ ]:
if canvas_source and canvas_source.result and canvas_source.result.get('source_id'):
    from nbinlineai.tools import stop_source
    stopped_source = stop_source(canvas_source.result['source_id'])
    print('Canvas stop requested:', stopped_source)
if canvas_saved and canvas_saved.media and canvas_saved.media.get('path'):
    from hashlib import sha256
    from pathlib import Path
    path = Path(canvas_saved.media['path'])
    expected = canvas_saved.media['sha256']
    if path.is_file() and path.name.startswith('capture-') and path.suffix == '.png' and sha256(path.read_bytes()).hexdigest() == expected:
        path.unlink()
        print('Removed exact example file:', path)
    else:
        print('Saved file was not deleted; inspect its server-root path manually:', canvas_saved.media['path'])

In [ ]:
print(stopped_source.status, stopped_source.result, stopped_source.error) if 'stopped_source' in globals() else print('No source stop needed.')